# Watcher - Performance Analysis in ComCam On-Sky Campaign

This analysis is part of the preparation for the **LSSTCam On-Sky Workshop**. The goal is to evaluate the performance of the **Watcher** during the **ComCam On-Sky campaign**, which took place from **October 24, 2024, to December 11, 2024**. 

This notebook focuses on a specific subtask: **Watcher - Response time**. What was the average response time to an alarm? This means that the alarm was either acknowledged or that the issue was resolved after it was triggered (or any level or “no alarm”). This can be a histogram or some sort of statistical metric. 

Laura Toribio 13-05-2025

In [ ]:
from astropy.time import Time

from lsst.sitcom.vandv.logger import create_logger
from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.summit.utils.tmaUtils import TMAEventMaker

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import pandas as pd


## Information about the Watcher

In [ ]:
# Create an EFD client instance
client = makeEfdClient()

In [ ]:
# make a list of all topics in the EFD related to Watcher
topics = await client.get_topics()
for topic in topics:
    if 'Watcher' in topic:
        print(topic)

In [ ]:
# get all fields related to the Watcher mute
await client.get_fields('lsst.sal.Watcher.logevent_alarm')

In [ ]:
# Get the duration data from October 24, 2024, to December 11, 2024.
start = Time("2024-10-24T00:00:00Z", scale="utc")
end = Time("2024-12-11T00:00:00Z", scale="utc")

watchers = await client.select_time_series(
                       "lsst.sal.Watcher.logevent_alarm", 
                      "*", 
                      start, 
                      end
)

# Watcher Response Time

In [ ]:
# Number of Watcher
num_watcher = len(watchers)
print(f"Number of watcher: {num_watcher}")

In [ ]:
# Convert Unix TAI timestamps to ISO 8601 format for better readability and datetime operations
watchers['private_efdStamp'] = pd.to_datetime(watchers['private_efdStamp'], unit="s")

watchers['timestampAcknowledged'] = pd.to_datetime(watchers['timestampAcknowledged'], unit="s")
watchers['timestampAutoAcknowledge'] = pd.to_datetime(watchers['timestampAutoAcknowledge'], unit="s")

NONE = 1
WARNING = 2
SERIOUS = 3
CRITICAL = 4

So let's discard the alarms with severity=NONE and those muted by some user.

In [ ]:
# Number of Watcher (filtered)
num_watcher = len(watcher_filtered)
print(f"Number of watcher after the filter: {num_watcher}")

In [ ]:
# Filter for severity 2,3,4. Not muted alarms and avoiding usual Enabled and ScriptQueue alarms.
watcher_filtered = watchers[
                  (watchers['severity'].isin([2, 3, 4])) & 
                  (watchers['mutedBy'] == '') & 
                  (~watchers['name'].str.contains('Enabled|ScriptFailed', case=False, regex=True))
]

Let's select the alarms that were acknowledged (by any user) and the time it took to respond. 


In [ ]:
# Select the alarms acknowledged by a user for severity 2,3,4
acknowledged_watchers = watcher_filtered[watcher_filtered['acknowledgedBy'].notna() & (watcher_filtered['acknowledgedBy'] != '')]

In [ ]:
# Number of Watcher (filtered with acknowledged by any user)
num_watcher = len(acknowledged_watchers)
print(f"Number of watcher after the filter: {num_watcher}")

In [ ]:
response_delta = acknowledged_watchers['timestampAcknowledged'] - acknowledged_watchers['private_efdStamp']

In [ ]:
#Filter null deltas
valid_delta_mask = (response_delta.notna())

# Apply mask and calculate response time in seconds
acknowledged_watchers = acknowledged_watchers[valid_delta_mask].copy()
acknowledged_watchers['responseTimeSeconds'] = response_delta[valid_delta_mask].dt.total_seconds()

In [ ]:
plt.figure(figsize=(10, 6))
acknowledged_watchers['responseTimeSeconds'].hist(bins=40, color='skyblue', edgecolor='black')
plt.xlabel('Response Time (seconds)')
plt.ylabel('Number of Alarms')
plt.title('Histogram of Alarm Response Times (Alarm acknowledged by users)')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
acknowledged_watchers['responseTimeSeconds'].hist(bins=40, color='skyblue', edgecolor='black')
plt.xlabel('Response Time (seconds)')
plt.ylabel('Number of Alarms')
plt.title('Histogram of Alarm Response Times (Alarm acknowledged by users)')
plt.ylim(0, 50)  # Limit Y-axis from 0 to 500
plt.grid(True)
plt.show()

In [ ]:
stats = acknowledged_watchers['responseTimeSeconds'].describe()
print(stats)

In [ ]:
# Labels
severity_labels = {2: 'Warning', 3: 'Serious', 4: 'Critical'}
acknowledged_watchers['severityLabel'] = acknowledged_watchers['severity'].map(severity_labels)

severity_order = ['Warning', 'Serious', 'Critical']
severity_colors = {
    'None': 'green',
    'Warning': 'yellow',
    'Serious': 'orange',
    'Critical': 'red'
}

summary = acknowledged_watchers.groupby('severityLabel')['responseTimeSeconds'].mean()
summary = summary.reindex(severity_order)

bar_colors = [severity_colors[sev] for sev in summary.index]

summary.plot(kind='bar', color=bar_colors, edgecolor='black')
plt.title('Average Response Time by Severity')
plt.ylabel('Average Response Time (seconds)')
plt.xlabel('Severity')
plt.grid(axis='y')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# sort
severity_order = ['Warning', 'Serious', 'Critical']

# boxplot
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=acknowledged_watchers,
    x='severityLabel',
    y='responseTimeSeconds',
    order=severity_order,
)

plt.title('Response Time by Alarm Severity')
plt.xlabel('Severity')
plt.ylabel('Response Time (seconds)')
plt.grid(True)
plt.tight_layout()
plt.show()

Now let's select those alarms that were automatically acknowledged.

In [ ]:
epoch_time = pd.Timestamp("1970-01-01 00:00:00.000000000")

# Select with the acknowledgedBy is nan and the data *is not 1970-01-01 00:00:00
auto_acknowledged = watcher_filtered[
    (watcher_filtered['acknowledgedBy'].isna() | (watcher_filtered['acknowledgedBy'] == ''))
     #& (watcher_filtered['timestampAutoAcknowledge']!= epoch_time)
     ]

In [ ]:
# Number of Watcher acknowledged
num_watcher_auto_ack = len(auto_acknowledged)
print(f"Number of watcher auto acknowledged: {num_watcher_auto_ack}")

In [ ]:
auto_acknowledged

In [ ]:
response_delta = auto_acknowledged['timestampAutoAcknowledge'] - auto_acknowledged['private_efdStamp']

In [ ]:
#Filter null deltas
valid_delta_mask = (response_delta.notna())

# Apply mask and calculate response time in seconds
auto_acknowledged = auto_acknowledged[valid_delta_mask].copy()
auto_acknowledged['responseTimeSeconds'] = response_delta[valid_delta_mask].dt.total_seconds()

In [ ]:
plt.figure(figsize=(10, 6))
auto_acknowledged['responseTimeSeconds'].hist(bins=40, color='skyblue', edgecolor='black')
plt.xlabel('Response Time (seconds)')
plt.ylabel('Number of Alarms')
plt.title('Histogram of Alarm Response Times (Auto Acknowledged)')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
auto_acknowledged['responseTimeSeconds'].hist(bins=40, color='skyblue', edgecolor='black')
plt.xlabel('Response Time (seconds)')
plt.ylabel('Number of Alarms')
plt.title('Histogram of Alarm Response Times ((Auto Acknowledged)')
plt.ylim(0, 20)  # Limit Y-axis from 0 to 500
plt.grid(True)
plt.show()

In [ ]:
stats = auto_acknowledged['responseTimeSeconds'].describe()
print(stats)

In [ ]:
# Labels
severity_labels = {1: 'None', 2: 'Warning', 3: 'Serious', 4: 'Critical'}
auto_acknowledged['severityLabel'] = auto_acknowledged['severity'].map(severity_labels)

severity_order = ['None', 'Warning', 'Serious', 'Critical']
severity_colors = {
    'None': 'green',
    'Warning': 'yellow',
    'Serious': 'orange',
    'Critical': 'red'
}

summary = auto_acknowledged.groupby('severityLabel')['responseTimeSeconds'].mean()
summary = summary.reindex(severity_order)

bar_colors = [severity_colors[sev] for sev in summary.index]

summary.plot(kind='bar', color=bar_colors, edgecolor='black')
plt.title('Average Response Time by Severity (Auto acknowledged)')
plt.ylabel('Average Response Time (seconds)')
plt.xlabel('Severity')
plt.grid(axis='y')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
severity_counts = auto_acknowledged['severityLabel'].value_counts().reindex(['None', 'Warning', 'Serious', 'Critical'], fill_value=0)

print(severity_counts)